In [13]:
import os
import sys
sys.path.append('..')
import torch
import librosa
from scipy.io import wavfile
from IPython import display as disp
from pathlib import Path
from tqdm import tqdm

from src.modules import ResynthesisModule, SpeakerModule

In [7]:

os.environ["CUDA_VISIBLE_DEVICES"] = "3" # Set the GPU device
MAX_WAV_VALUE = 32768.0

In [4]:
speaker_encoder = SpeakerModule(device="cuda")
resynthesizer = ResynthesisModule.load_from_checkpoint(
    "/mnt/nvme-data1/waris/PSI-TAMU/new_repo/Xarts/logs/xarts/resynth_no_la_r3/checkpoints/last_v2.ckpt",
    encoder_path=None,
    map_location="cuda"
)
resynthesizer.eval()

INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/spkrec-xvect-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch custom.py: Fetching from HuggingFace Hub 'speechbrain/spkrec-xvect-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch embedding_model.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-xvect-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch mean_var_norm_emb.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-xvect-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch classifier.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-xvect-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch label_encoder.txt: Fetching from HuggingFace Hub 'speechbrain/spkrec-xvect-voxceleb' if not cached
INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: embedding_model, mean_var_norm_emb, classifier, label_encoder
INFO:speechbrain.utils.fetching:Fetch hyperpara

ResynthesisModule(
  (encoder): PasrModule(
    (net): HiFiEncoder(
      (conv_pre): CausalConv(
        (layer): ParametrizedRawStreamingConv1d(
          1, 16, kernel_size=(7,), stride=(1,)
          (parametrizations): ModuleDict(
            (weight): ParametrizationList(
              (0): _WeightNorm()
            )
          )
        )
      )
      (downs): ModuleList(
        (0): CausalConv(
          (layer): ParametrizedRawStreamingConv1d(
            16, 32, kernel_size=(4,), stride=(2,)
            (parametrizations): ModuleDict(
              (weight): ParametrizationList(
                (0): _WeightNorm()
              )
            )
          )
        )
        (1): CausalConv(
          (layer): ParametrizedRawStreamingConv1d(
            32, 64, kernel_size=(4,), stride=(2,)
            (parametrizations): ModuleDict(
              (weight): ParametrizationList(
                (0): _WeightNorm()
              )
            )
          )
        )
        (2): 

In [8]:
wav_file = "/mnt/nvme-data1/waris/dummy_data/BDL/arctic_a0001.wav"
_device = "cuda"
sample_rate = 16000

audio, _ = librosa.load(wav_file, sr=sample_rate)

disp.Audio(audio, rate=16000)

In [9]:
audio = torch.from_numpy(audio).unsqueeze(0).unsqueeze(0).to(_device)
embed = speaker_encoder(audio.squeeze(0))
resyth_audio = resynthesizer(audio, embed)
resyth_audio = resyth_audio.squeeze()
resyth_audio = resyth_audio * MAX_WAV_VALUE
resyth_audio = resyth_audio.detach().cpu().numpy().astype('int16')

disp.Audio(resyth_audio, rate=16000)

In [16]:
resynthesizer.eval()
with torch.no_grad():
    original_length = audio.shape[-1]
    # Pad audio for lookahead
    # frame_size = int(resynthesizer_mm.hparams.sampling_rate // resynthesizer_mm.hparams.frame_rate)
    frame_size = 320

    # frame_size = int(encoder_mm.net.sample_rate // encoder_mm.net.frame_rate)
    all_audios = []
    all_z_embeds = []

    with resynthesizer.encoder.net.streaming(batch_size=1):
        with resynthesizer.decoder.streaming(batch_size=1):
            for offset in tqdm(range(0, original_length, frame_size)):
                frame = audio[:, :, offset:offset + frame_size]
                _, z_embed = resynthesizer.encoder.net(frame)
                if z_embed.shape[-1] == 0:
                    continue
                resynth_audio_chunk = resynthesizer.decoder.decode(z_embed, embed)
                all_audios.append(resynth_audio_chunk)   
                all_z_embeds.append(z_embed)    
    
all_audios = torch.cat(all_audios, dim=-1)
resyth_audio = all_audios.squeeze().cpu().numpy()
resyth_audio = resyth_audio * MAX_WAV_VALUE
resyth_audio = resyth_audio.astype('int16')

disp.Audio(resyth_audio, rate=16000)

100%|██████████| 162/162 [00:07<00:00, 20.28it/s]
